### 07 - Comparaison de modeles de classification
#### HumanForYou - Attrition ML

Objectif: comparer plusieurs modeles sur les memes donnees preparees en 05 (suite logique de 01->04).
Aucun nouveau nettoyage ou feature engineering metier n'est applique ici.

- **Entrees**: `data/processed/attrition_train_prepared.csv`, `data/processed/attrition_test_prepared.csv`
- **Sorties**: `data/processed/attrition_model_comparison.csv`, `data/processed/attrition_model_predictions_test.csv`, `data/processed/attrition_model_cv_scores.csv`

#### 1. Imports

Cette section prepare les composants necessaires pour comparer plusieurs familles de modeles:
- lineaire, arbres/boosting, SVM, k-NN,
- metriques de classification,
- validation croisee stratifiee.

L'objectif est de comparer performance et robustesse sur une base commune.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

#### 2. Chargement

On recharge exactement les memes donnees preparees en 05 et on reconstruit `X`/`y` pour train et test.

Cette coherence de donnees est indispensable pour que la comparaison entre modeles soit juste.


In [ ]:
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
train_path = os.path.join(PROCESSED_DIR, 'attrition_train_prepared.csv')
test_path = os.path.join(PROCESSED_DIR, 'attrition_test_prepared.csv')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

X_train = train_df.drop(columns=['Attrition']).copy()
y_train = train_df['Attrition'].astype(int).copy()
X_test = test_df.drop(columns=['Attrition']).copy()
y_test = test_df['Attrition'].astype(int).copy()

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

#### 3. Cross-validation

Chaque modele est evalue en validation croisee stratifiee (5 folds) sur le train uniquement.

On suit deux indicateurs complementaires:
- `F1` pour l'equilibre precision/recall,
- `ROC-AUC` pour la capacite de separation globale.

La moyenne et l'ecart-type permettent de juger la stabilite des performances.


In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'RandomForestClassifier': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced_subsample', n_jobs=-1),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC_RBF': SVC(kernel='rbf', C=3.0, gamma='scale', probability=True, random_state=42),
    'KNeighborsClassifier': KNeighborsClassifier(n_neighbors=15),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
for model_name, model in models.items():
    f1_scores = cross_val_score(model, X_train, y_train, scoring='f1', cv=cv, n_jobs=1)
    auc_scores = cross_val_score(model, X_train, y_train, scoring='roc_auc', cv=cv, n_jobs=1)
    cv_rows.append({
        'model': model_name,
        'f1_cv_mean': float(f1_scores.mean()),
        'f1_cv_std': float(f1_scores.std()),
        'roc_auc_cv_mean': float(auc_scores.mean()),
        'roc_auc_cv_std': float(auc_scores.std()),
    })

cv_df = pd.DataFrame(cv_rows).sort_values('f1_cv_mean', ascending=False).reset_index(drop=True)
cv_df

#### 4. Evaluation test

Apres la phase CV, chaque modele est entraine sur tout le train puis evalue sur le test hold-out.

Le notebook harmonise la sortie probabiliste:
- `predict_proba` quand disponible,
- approximation via `decision_function` sinon.

On obtient ainsi un tableau de comparaison final sur les memes metriques pour tous les modeles.


In [ ]:
comparison_rows = []
predictions_df = pd.DataFrame({'y_true': y_test})

for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        raw = model.decision_function(X_test)
        y_proba = 1 / (1 + np.exp(-raw))

    predictions_df[f'y_proba_{model_name}'] = y_proba
    predictions_df[f'y_pred_{model_name}'] = y_pred

    comparison_rows.append({
        'model': model_name,
        'accuracy_test': float(accuracy_score(y_test, y_pred)),
        'precision_test': float(precision_score(y_test, y_pred, zero_division=0)),
        'recall_test': float(recall_score(y_test, y_pred, zero_division=0)),
        'f1_test': float(f1_score(y_test, y_pred, zero_division=0)),
        'roc_auc_test': float(roc_auc_score(y_test, y_proba)),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values('f1_test', ascending=False).reset_index(drop=True)
comparison_df

#### 5. Export + figures

Les tableaux de comparaison (CV, test, predictions) sont sauvegardes dans `data/processed`.
Deux visualisations de synthese sont produites:
- classement des modeles par F1 test,
- courbe ROC du meilleur modele retenu.

Ces sorties servent de base a la decision de selection du modele final.


In [ ]:
FIG_DIR = os.path.join('..', 'reports', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

comparison_path = os.path.join(PROCESSED_DIR, 'attrition_model_comparison.csv')
predictions_path = os.path.join(PROCESSED_DIR, 'attrition_model_predictions_test.csv')
cv_path = os.path.join(PROCESSED_DIR, 'attrition_model_cv_scores.csv')

comparison_df.to_csv(comparison_path, index=False)
predictions_df.to_csv(predictions_path, index=False)
cv_df.to_csv(cv_path, index=False)

plot_df = comparison_df.sort_values('f1_test', ascending=True)
plt.figure(figsize=(9, 5))
plt.barh(plot_df['model'], plot_df['f1_test'])
plt.xlabel('F1 score (test)')
plt.title('Comparaison des modeles - Attrition')
plt.tight_layout()
f1_fig_path = os.path.join(FIG_DIR, 'attrition_models_f1_test.png')
plt.savefig(f1_fig_path, dpi=300)
plt.close()

best_model_name = comparison_df.iloc[0]['model']
best_proba = predictions_df[f'y_proba_{best_model_name}']
fpr, tpr, _ = roc_curve(y_test, best_proba)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Best model: {best_model_name}')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC curve - Best model')
plt.legend()
plt.tight_layout()
roc_fig_path = os.path.join(FIG_DIR, 'attrition_best_model_roc_curve.png')
plt.savefig(roc_fig_path, dpi=300)
plt.close()

print(f'Saved: {comparison_path}')
print(f'Saved: {predictions_path}')
print(f'Saved: {cv_path}')
print(f'Saved: {f1_fig_path}')
print(f'Saved: {roc_fig_path}')